In [1]:
# Reload final model and data (self-contained)

import pandas as pd
import joblib

# Reload cleaned data
df = pd.read_csv("../data/raw/genetic_disorder_raw.csv")
identity_cols = ["Patient Id", "Patient First Name", "Family Name", "Father's name"]
df_clean = df.drop(columns=identity_cols)

numeric_cols = df_clean.select_dtypes(include=["float64", "int64"]).columns
categorical_cols = df_clean.select_dtypes(include="object").columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in categorical_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Load your saved final model
model = joblib.load("../model/registry/model_v1.pkl")
model 


,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [2]:
# Check global feature importance (overall, not per-patient yet)

importances = pd.Series(model.feature_importances_, index=model.feature_names_in_)
importances.sort_values(ascending=False).head(10)

Symptom_Count                                       0.116528
Blood cell count (mcL)                              0.083706
White Blood cell count (thousand per microliter)    0.076555
Parent_Age_Gap                                      0.069808
Father's age                                        0.062640
Mother's age                                        0.061523
Patient Age                                         0.060686
Symptom 5                                           0.035458
H/O radiation exposure (x-ray)                      0.033932
H/O substance abuse                                 0.033756
dtype: float64

In [3]:
!pip install shap 

In [ ]:
# Confirm SHAP now 
import shap 

explainer = shap.TreeExplainer(model)
explainer 

In [7]:
from sklearn.preprocessing import LabelEncoder

# Rebuild engineered features
symptom_cols = ["Symptom 1", "Symptom 2", "Symptom 3", "Symptom 4", "Symptom 5"]
df_clean["Symptom_Count"] = df_clean[symptom_cols].sum(axis=1)
df_clean["Parent_Age_Gap"] = df_clean["Father's age"] - df_clean["Mother's age"]

# Label encode categoricals (matches how final_rf_fe was actually trained)
df_encoded = df_clean.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

X = df_encoded.drop(columns=["Genetic Disorder", "Disorder Subclass"])
X.shape

(18000, 29)

In [8]:
# Pick one sample patient and get their SHAP values
sample_patient = X.iloc[[0]] # first patient, as a test
shap_values = explainer.shap_values(sample_patient)

# shap_values shape check 
import numpy as np 
np.array(shap_values).shape 

(1, 29, 3)

In [9]:
# Find out which class this patient was actually predicted as 
predicted_class = model.predict(sample_patient)[0]
predicted_class

np.int64(0)

In [10]:
# Get the top 3 features that drove THAT specific prediction
class_index = list(model.classes_).index(predicted_class)
patient_shap = shap_values[0, :, class_index]

feature_impact = pd.Series(patient_shap, index=X.columns)
top_3 = feature_impact.abs().sort_values(ascending=False).head(3)
top_3

Symptom_Count             0.130939
Parent_Age_Gap            0.041282
Genes in mother's side    0.029836
dtype: float64

In [ ]:
# Turn this into a reusable function for any patient, not just row 0
def get_top_features(patient_row, model, explainer, X_columns, top_n=3):
    shap_values = explainer.shap_values(patient_row)
    predicted_class = model.predict(patient_row)[0]
    class_index = list(model.classes_).index(predicted_class)
    
    patient_shap = shap_values[0, :, class_index]
    feature_impact = pd.Series(patient_shap, index=X_columns)
    top_features = feature_impact.abs().sort_values(ascending=False).head(top_n)
    
    return {
        "predicted_class": predicted_class,
        "top_features": top_features.to_dict()
    }

In [12]:
# Test the function on a different patient (row 5) to confirm it works generally, not just for row 0
test_result = get_top_features(X.iloc[[5]], model, explainer, X.columns)
test_result

{'predicted_class': np.int64(1),
 'top_features': {'Symptom_Count': 0.1760927807057732,
  'Symptom 5': 0.07778582840140168,
  "Genes in mother's side": 0.06457478777598075}}

In [13]:
# Map raw column names to human-readable labels
label_map = {
    "Symptom_Count": "Number of reported symptoms",
    "Parent_Age_Gap": "Age gap between parents",
    "Genes in mother's side": "Family history on mother's side",
    "Father's age": "Father's age",
    "Mother's age": "Mother's age",
    "Blood cell count (mcL)": "Blood cell count",
    "White Blood cell count (thousand per microliter)": "White blood cell count",
    "Patient Age": "Patient's age",
    "Symptom 1": "Symptom 1 present",
    "Symptom 2": "Symptom 2 present",
    "Symptom 3": "Symptom 3 present",
    "Symptom 4": "Symptom 4 present",
    "Symptom 5": "Symptom 5 present",
}

def humanize_features(top_features_dict):
    return {label_map.get(k, k): v for k, v in top_features_dict.items()}

humanize_features(test_result["top_features"])

{'Number of reported symptoms': 0.1760927807057732,
 'Symptom 5 present': 0.07778582840140168,
 "Family history on mother's side": 0.06457478777598075}

In [14]:
# Map the predicted class number back to its real label
disorder_labels = {
    0: "Mitochondrial genetic inheritance disorder",
    1: "Multifactorial genetic inheritance disorder",
    2: "Single-gene inheritance disease"
}
disorder_labels[test_result["predicted_class"]]

'Multifactorial genetic inheritance disorder'